#### LoRA from scratch, verified against PEFT

In [2]:
#pip install peft

   ---------------------------------------- 0.0/835.4 kB ? eta -:--:--
   ---------------------------------------- 835.4/835.4 kB 10.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import math
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
import lab06_starter as S


In [8]:
#import inspect
#print(inspect.getsource(run_piece2))

def run_piece2():
    print("--- LoRA from Scratch & Verification ---")

    # 1. Test single layer equivalence at init
    linear = nn.Linear(64, 32)
    lora_layer = LoRALinear(linear, r=4)
    x = torch.randn(2, 64)

    with torch.no_grad():
        diff = (lora_layer(x) - linear(x)).abs().max().item()
    print(f"Difference between base layer and LoRA layer at init (B=0): {diff:.2e}")
    assert diff < 1e-6, "LoRA layer does not match the base layer at initialization!!!"

    # 2. Parameter count check for one layer
    trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
    expected_params = 4 * (64 + 32)
    print(f"Trainable params: {trainable_params} (Expected r * (in + out) = {expected_params})")
    assert trainable_params == expected_params

    # 3. Merged weight check
    with torch.no_grad():
        # Give B non-zero values
        lora_layer.lora_B.fill_(0.1)
        x_test = torch.randn(4, 64)
        out_two_path = lora_layer(x_test)

In [10]:
# 4. Compare against PEFT library on full model
#model_custom, tok = S.load_model()
#replace_linear_with_lora(model_custom, r=8, alpha=16)

#model_peft, _ = S.load_model()
#model_peft = S.apply_lora(model_peft, r=8, alpha=16)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [6]:
#%who

LoRALinear	 LoraConfig	 S	 dataframe_columns	 dataframe_hash	 dtypes_str	 get_dataframes	 get_peft_model	 getpass	 
hashlib	 import_pandas_safely	 is_data_frame	 json	 math	 nn	 replace_linear_with_lora	 run_piece2	 torch	 



In [12]:
#model_custom, _ = S.load_model()
#replace_linear_with_lora(model_custom, r=8, alpha=16)

#model_peft, _ = S.load_model()
#model_peft = S.apply_lora(model_peft, r=8, alpha=16)

#custom_trainable, _ = S.count_params(model_custom)
#peft_trainable, _ = S.count_params(model_peft)

#print(f"Custom: {custom_trainable} | PEFT: {peft_trainable} | Difference: {custom_trainable - peft_trainable}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Custom: 121704768 | PEFT: 460800 | Difference: 121243968


In [15]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int, alpha: float = None):
        super().__init__()
        self.base_layer = base_layer
        self.r = r
        self.alpha = alpha if alpha is not None else 2 * r
        self.scaling = self.alpha / self.r

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        # Freeze base layer weights
        for p in self.base_layer.parameters():
            p.requires_grad = False

        # LoRA matrices: A is small random (Gaussian), B is zero
        self.lora_A = nn.Parameter(torch.zeros(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        
        # Initialize A using Kaiming/Gaussian uniform, B to zero
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base_layer(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

def replace_linear_with_lora(model, r, alpha, target_modules=["q_proj", "v_proj"]):
    # 1. Freeze the entire base model first
    for param in model.parameters():
        param.requires_grad = False

    # 2. Recursively find and replace target linear layers
    for name, module in model.named_children():
        if isinstance(module, nn.Linear) and any(t in name for t in target_modules):
            setattr(model, name, LoRALinear(module, r=r, alpha=alpha))
        else:
            replace_linear_with_lora(module, r=r, alpha=alpha)

def run_piece2():
    print("--- LoRA from Scratch & Verification ---")
    
    # 1. Test single layer equivalence at init
    linear = nn.Linear(64, 32)
    lora_layer = LoRALinear(linear, r=4)
    x = torch.randn(2, 64)
    
    with torch.no_grad():
        diff = (lora_layer(x) - linear(x)).abs().max().item()
    print(f"Difference between base layer and LoRA layer at init (B=0): {diff:.2e}")
    assert diff < 1e-6, "LoRA layer does not match the base layer at initialization!!!"

    # 2. Parameter count check for one layer
    trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
    expected_params = 4 * (64 + 32)
    print(f"Trainable params: {trainable_params} (Expected r * (in + out) = {expected_params})")
    assert trainable_params == expected_params

    # 3. Merged weight check
    with torch.no_grad():
        # Give B non-zero values
        lora_layer.lora_B.fill_(0.1)
        x_test = torch.randn(4, 64)
        out_two_path = lora_layer(x_test)
        
        # Manual merge: W_merged = W_base + scaling * (B @ A)
        w_base = lora_layer.base_layer.weight
        w_delta = (lora_layer.lora_B @ lora_layer.lora_A) * lora_layer.scaling
        w_merged = w_base + w_delta
        out_merged = x_test @ w_merged.T + lora_layer.base_layer.bias
        
        merge_diff = (out_two_path - out_merged).abs().max().item()
    print(f"Maximum difference between two-path and merged weight pass is: {merge_diff:.2e}")
    assert merge_diff < 1e-5

    # 4. Compare against PEFT library on full model
    model_custom, tok = S.load_model()
    replace_linear_with_lora(model_custom, r=8, alpha=16)
    
    model_peft, _ = S.load_model()
    model_peft = S.apply_lora(model_peft, r=8, alpha=16)

    custom_trainable, custom_total = S.count_params(model_custom)
    peft_trainable, peft_total = S.count_params(model_peft)
    print(f"Custom LoRA trainable params: {custom_trainable:,} | PEFT LoRA trainable params: {peft_trainable:,}")
    assert custom_trainable == peft_trainable, "Trainable parameter count mismatch with PEFT!!!"

    # 5. Print parameter counts and percentages for ranks 1, 4, 16
    for r in [1, 4, 16]:
        m, _ = S.load_model()
        replace_linear_with_lora(m, r=r, alpha=2*r)
        trainable, total = S.count_params(m)
        pct = (trainable / total) * 100
        print(f"Rank {r:2d}: Trainable = {trainable:,} | Total = {total:,} | Percentage = {pct:.3f}%")

if __name__ == "__main__":
    run_piece2()

--- LoRA from Scratch & Verification ---
Difference between base layer and LoRA layer at init (B=0): 0.00e+00
Trainable params: 384 (Expected r * (in + out) = 384)
Maximum difference between two-path and merged weight pass is: 4.77e-07


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Custom LoRA trainable params: 460,800 | PEFT LoRA trainable params: 460,800


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Rank  1: Trainable = 57,600 | Total = 134,572,608 | Percentage = 0.043%


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Rank  4: Trainable = 230,400 | Total = 134,745,408 | Percentage = 0.171%


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Rank 16: Trainable = 921,600 | Total = 135,436,608 | Percentage = 0.680%
